# Day 27: Packages, uv and Project Environments

**Today's question:** how do real projects use code from the internet without breaking each other, and make
the program run the same on every computer?

Most of today happens in the **terminal** (see the steps in the lesson). This notebook shows what is going on
behind the scenes. Before running a cell marked 🤔, predict what it will show.

## 1. Which Python is running this notebook?

`sys` is a standard module that knows about the running Python itself.

In [ ]:
import sys

print(sys.version)

In [ ]:
print(sys.executable)

Look at the path: it ends in **`.venv/bin/python`** (Windows: `.venv\Scripts\python.exe`). This notebook does
not use "the" Python on your computer. It uses the course's own **virtual environment**, the `.venv` folder
that uv created in the course folder on Day 01.

## 2. Where do modules come from?

Every imported module has a `__file__`: the file it was loaded from.

In [ ]:
import random

print(random.__file__)

🤔 **Predict:** `pytest` is not part of Python; uv installed it for this course. Where will its file be?

In [ ]:
import pytest

print(pytest.__file__)

- **Standard library** modules (`random`, `math`, `datetime`) come with Python itself.
- **Third-party** packages (`pytest`, `numpy`, `pandas`) live inside **`.venv/lib/.../site-packages`**.
  They belong to this project only.

`importlib.metadata.version` tells you which version of a package is installed:

In [ ]:
from importlib.metadata import version

print("pytest", version("pytest"))
print("numpy ", version("numpy"))
print("pandas", version("pandas"))

## 3. A package that is not installed
`rich` (a package that prints colourful tables) is **not** part of this course environment.

In [ ]:
import rich

The fix is **not** typed in the notebook. In the terminal, inside the right project, you run `uv add rich`.
In the lesson we do that in a brand-new project of your own, so the course environment stays unchanged.

## 4. Why every project gets its own environment

Imagine two projects on Sara's laptop:

```
expense-app/          needs pandas 2.2
  .venv/   ← pandas 2.2 lives here
old-thesis-code/      only works with pandas 1.5
  .venv/   ← pandas 1.5 lives here
```

If both used one shared Python, installing pandas 2.2 for the new project would break the old one.
A **virtual environment** (`.venv`) is a private folder of packages for *one* project.

uv manages it with two files:

| File | Written by | Contains |
|------|------------|----------|
| `pyproject.toml` | you (via `uv add`) | the packages **you asked for**, e.g. `"rich>=15.0.0"` |
| `uv.lock` | uv, never by hand | the **exact** version of *every* package that was installed, including the packages they need |

Share `pyproject.toml` and `uv.lock` with your team. Never share `.venv`: anyone can rebuild it with `uv sync`.

## 5. Reading this course's own project files

The course folder is itself a uv project. Its `pyproject.toml` is three folders up.

In [ ]:
from pathlib import Path

course_root = Path("../../..")
print((course_root / "pyproject.toml").read_text(encoding="utf-8"))

🤔 **Predict:** `pyproject.toml` lists 8 packages under `dependencies`. How many packages does `uv.lock` pin?
Each one starts with a line `[[package]]`.

In [ ]:
lock_lines = (course_root / "uv.lock").read_text(encoding="utf-8").split("\n")
package_starts = [line for line in lock_lines if line == "[[package]]"]
print(len(package_starts), "packages are pinned in uv.lock")

Many more than 8, because packages need other packages (*dependencies of dependencies*). `uv.lock` records all of
them so every student gets **exactly** the same versions. Here is the `pytest` entry:

In [ ]:
start = lock_lines.index('name = "pytest"')
for line in lock_lines[start - 1:start + 10]:
    print(line[:80])

It records the exact version, where it was downloaded from, and which other packages pytest itself needs.

## 6. Packages: a folder of modules

When a project grows, related modules are grouped into a **package**: a folder with an `__init__.py` file inside.
This folder contains one called `shop`:

```
day-27-packages-uv/
├── demo.ipynb
└── shop/
    ├── __init__.py   ← marks "shop" as a package; runs on first import
    ├── prices.py     ← add_tax, apply_discount, format_price
    └── stock.py      ← is_low, restock_list
```

(The word "package" means two related things: a folder of modules like this, and something you install with
`uv add`, which is a package that someone published.)

In [ ]:
import shop

print(shop.SHOP_NAME)

To reach a module *inside* the package, use a dot: `package.module`.

In [ ]:
from shop import prices

print(prices.add_tax(10))
print(prices.format_price(1250))

In [ ]:
from shop.prices import apply_discount

print(apply_discount(80, 25))

🤔 **Predict:** `restock_list` returns items whose quantity is below 5, sorted A–Z. What does it print?

In [ ]:
import shop.stock

shelf = {"milk": 3, "bread": 12, "eggs": 4, "rice": 20}
print(shop.stock.restock_list(shelf))

| Import | Then call |
|--------|-----------|
| `import shop.stock` | `shop.stock.restock_list(...)` |
| `from shop import prices` | `prices.add_tax(...)` |
| `from shop.prices import apply_discount` | `apply_discount(...)` |

## 7. Common mistakes (these cells fail **on purpose**)

In [ ]:
# Mistake 1: prices.py is inside the shop folder, so "prices" alone is not found
import prices

In [ ]:
# Mistake 2: add_tax lives in shop/prices.py, not in shop/__init__.py
from shop import add_tax

Terminal mistakes (no notebook cell can show these):

| Mistake | What happens | Do this instead |
|---------|--------------|-----------------|
| `pip install rich` | installs into the wrong Python, or fails | `uv add rich` inside the project folder |
| `python main.py` | may use a Python without your packages | `uv run main.py` |
| `uv init` inside the course folder | uv adds it to the *course* project | create projects outside the course folder |
| Editing `uv.lock` by hand | uv overwrites or rejects it | change `pyproject.toml` via `uv add` / `uv remove` |
| Sending someone your `.venv` | huge, and only works on your machine | share `pyproject.toml` + `uv.lock`; they run `uv sync` |

## 8. Real-world example: a receipt built from the package

In [ ]:
from shop import prices, stock

cart = [("notebook", 3.50, 4), ("pen", 1.20, 10), ("backpack", 42.00, 1)]
shelf = {"notebook": 6, "pen": 25, "backpack": 2}

subtotal = 0
for item, price, quantity in cart:
    line_total = price * quantity
    subtotal += line_total
    print(f"{item:<10} {quantity:>3} x {price:>6.2f} = {prices.format_price(line_total):>8}")

after_discount = prices.apply_discount(subtotal, 10)
print(f"{'Subtotal':<24} {prices.format_price(subtotal):>8}")
print(f"{'After 10% discount':<24} {prices.format_price(after_discount):>8}")
print(f"{'Total with tax':<24} {prices.format_price(prices.add_tax(after_discount)):>8}")
print("Reorder soon:", stock.restock_list(shelf))

## Recap

1. Each project has its own **virtual environment** (`.venv`), so projects cannot break each other.
2. `uv init` creates a project, `uv add` installs a package and records it, `uv run` runs code inside the
   environment, and `uv sync` rebuilds `.venv` from the lock file.
3. `pyproject.toml` lists what you asked for; `uv.lock` pins the exact version of everything installed.
4. A **package** is a folder of modules with an `__init__.py`; import from it with dots: `from shop import prices`.

**Next time:** when code goes wrong, using tracebacks and the VS Code debugger to find the bug.